# Atelier Python Session 6 (suite) : le traitement automatique des langues avec spaCy

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stage-to-data/hackspace-shs/blob/main/cours-python-resources/session-6/Atelier%20Python%20Session%206%20-%20NLP%20-%20spaCy.ipynb)

Dans le premier notebook de cette séance, nous avons construit un pipeline de TAL étape par étape avec `nltk`. Nous
allons maintenant faire la même chose avec [spaCy](https://spacy.io/), un package plus récent, très utilisé aujourd'hui,
qui a plusieurs avantages :
- il fait **toutes les étapes d'un coup** : tokenisation, POS tagging, lemmatisation, entités nommées... ;
- ses modèles sont plus performants que ceux de `nltk` ;
- il propose des modèles pour de nombreuses langues, **dont le français**.

Puis nous verrons deux autres analyses courantes : l'**analyse de sentiment** et la **modélisation thématique** (*topic
modelling*). Enfin, nous rassemblerons tout cela dans une **classe**, comme vu en séance 5.

## Préparation

spaCy est installé dans Colab, mais il a besoin d'un **modèle** pour chaque langue. Les modèles `sm` (*small*) sont
légers et rapides ; il existe aussi des versions `md`, `lg` et `trf`, plus lourdes mais plus précises. Téléchargeons le
modèle anglais et le modèle français :

In [ ]:
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm

In [ ]:
import spacy

# On charge le modèle anglais (cela peut prendre quelques secondes) :
nlp = spacy.load("en_core_web_sm")

texte = ("In December 1720, a great fire burned for almost a week in Rennes, the capital of Brittany. "
         "Much of the medieval city centre was destroyed, and thousands of people lost their homes. "
         "The engineer Isaac Robelin, and later the architect Jacques Gabriel, drew up plans to rebuild the city "
         "with wide, straight streets and elegant stone buildings. "
         "Today, visitors can still see the contrast between the old timber-framed houses "
         "and the classical squares of the eighteenth century.")

## 1. Le document spaCy

Avec spaCy, on donne le texte au modèle, comme à une fonction, et on obtient un objet `Doc`, qui contient déjà le
résultat de **toutes** les analyses. Un `Doc` se parcourt comme une liste de tokens, et son attribut `.sents` donne les
phrases :

In [ ]:
doc = nlp(texte)

print(type(doc))

mots = []
for token in doc:
  mots.append(token.text)
print(mots)

for phrase in doc.sents:
  print("-", phrase.text)

## 2. Les attributs des tokens

Chaque token est un objet, avec de nombreux attributs. Voici les plus utiles (remarquez le tiret bas à la fin de
certains : sans lui, spaCy renvoie un identifiant numérique au lieu du texte) :

| Attribut | Contenu |
|---|---|
| `token.text` | le texte du token |
| `token.lemma_` | son lemme |
| `token.pos_` | sa catégorie grammaticale simplifiée (`NOUN`, `VERB`, `ADJ`, `PROPN`...) |
| `token.tag_` | sa catégorie grammaticale détaillée (les étiquettes du Penn Treebank, comme avec `nltk`) |
| `token.is_stop` | `True` si c'est un mot vide |
| `token.is_alpha` | `True` s'il n'est composé que de lettres |
| `token.is_punct` | `True` si c'est de la ponctuation |

In [ ]:
for token in doc:
  print(token.text, "|", token.lemma_, "|", token.pos_, "|", token.tag_, "|", token.is_stop, "|", token.is_alpha)

Si une étiquette vous est inconnue, `spacy.explain()` vous l'explique :

In [ ]:
print(spacy.explain("PROPN"))
print(spacy.explain("VBD"))

## 3. Lemmes, mots vides et fréquences

Plus besoin de traduire les étiquettes pour le lemmatiseur comme avec `nltk` : spaCy a déjà tout fait. Pour obtenir les
lemmes des mots « pleins » du texte, il suffit de filtrer avec `is_stop` et `is_alpha` :

In [ ]:
lemmes = []
for token in doc:
  if token.is_stop == False and token.is_alpha:
    lemmes.append({"original": token.text, "lemme": token.lemma_})

print(lemmes)

# Et comptons les lemmes, comme en séance 3 :
compteur = {}
for element in lemmes:
  lemme = element["lemme"].lower()
  if lemme in compteur:
    compteur[lemme] = compteur[lemme] + 1
  else:
    compteur[lemme] = 1

print(compteur)

## 4. POS tagging

In [ ]:
pos_tags = []
for token in doc:
  if token.is_stop == False and token.is_alpha:
    pos_tags.append({"mot": token.text, "pos": token.pos_, "tag": token.tag_})

print(pos_tags)

## 5. Les entités nommées

Les entités sont dans l'attribut `.ents` du document. Chacune a un texte et un type (`.label_`). Les types de spaCy sont
un peu différents de ceux de `nltk` : `PERSON`, `GPE` (pays, villes...), `LOC` (autres lieux), `ORG`, `DATE`, `NORP`
(nationalités, groupes religieux ou politiques)...

In [ ]:
entites = []
for entite in doc.ents:
  entites.append({"texte": entite.text, "type": entite.label_})
  print(entite.text, ":", entite.label_, "-", spacy.explain(entite.label_))

spaCy propose aussi `displacy`, un outil pour visualiser les entités directement dans le texte :

In [ ]:
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

## 6. Et en français !

Pour analyser un texte français, il suffit de charger le modèle français : tout le reste du code est identique.

In [ ]:
nlp_fr = spacy.load("fr_core_news_sm")

texte_fr = ("En décembre 1720, un grand incendie ravagea Rennes pendant près d'une semaine. "
            "L'ingénieur Isaac Robelin, puis l'architecte Jacques Gabriel, dessinèrent les plans de la nouvelle ville. "
            "Aujourd'hui, la place du Parlement de Bretagne témoigne encore de cette reconstruction.")

doc_fr = nlp_fr(texte_fr)

for token in doc_fr:
  if token.is_alpha and not token.is_stop:
    print(token.text, "->", token.lemma_, "(" + token.pos_ + ")")

print("--")
for entite in doc_fr.ents:
  print(entite.text, ":", entite.label_)

displacy.render(doc_fr, style="ent", jupyter=True)

Remarquez que le modèle français utilise d'autres types d'entités : `PER` (personne), `LOC` (lieu), `ORG`
(organisation) et `MISC` (divers). Et que les modèles `sm` font parfois des erreurs : n'hésitez pas à essayer
`fr_core_news_md` ou `fr_core_news_lg` pour de meilleurs résultats.

## 7. L'analyse de sentiment

L'**analyse de sentiment** cherche à mesurer l'opinion ou l'émotion exprimée dans un texte. Nous allons utiliser
[TextBlob](https://textblob.readthedocs.io/), qui donne deux scores :
- la **polarité** (*polarity*), entre -1 (très négatif) et 1 (très positif) ;
- la **subjectivité** (*subjectivity*), entre 0 (très objectif, factuel) et 1 (très subjectif, opinion personnelle).

TextBlob repose sur un dictionnaire de mots annotés (*good* est positif, *terrible* négatif...). C'est simple et rapide,
mais limité : il ne comprend ni l'ironie, ni le contexte, et ne fonctionne qu'en anglais (il existe une extension pour
le français, `textblob-fr`).

In [ ]:
!pip install -q textblob

In [ ]:
from textblob import TextBlob

blob = TextBlob(texte)
print("Polarité :", blob.sentiment.polarity)
print("Subjectivité :", blob.sentiment.subjectivity)

# Comparons quelques phrases :
exemples = [
  "The new library is wonderful and the staff are very helpful.",
  "The archive was closed and the catalogue is a terrible mess.",
  "The archive opens at nine o'clock."
]
for exemple in exemples:
  sentiment = TextBlob(exemple).sentiment
  print(f"{sentiment.polarity:+.2f} | {sentiment.subjectivity:.2f} | {exemple}")

*(Dans la f-string ci-dessus, `:+.2f` est une indication de format : on affiche le nombre avec son signe et deux
chiffres après la virgule.)*

## 8. La modélisation thématique (topic modelling)

La **modélisation thématique** cherche à découvrir automatiquement les grands **thèmes** d'un corpus de documents.
L'algorithme le plus connu est [LDA](https://fr.wikipedia.org/wiki/Allocation_de_Dirichlet_latente) (*Latent Dirichlet
Allocation*) : il part du principe que chaque document est un mélange de thèmes, et que chaque thème est caractérisé par
des mots qui apparaissent souvent ensemble. On lui donne le nombre de thèmes à chercher, et il renvoie, pour chaque
thème, les mots les plus représentatifs. C'est ensuite à nous d'interpréter ces listes de mots !

LDA a besoin de **plusieurs documents** (idéalement des centaines, voire des milliers) pour trouver des régularités.
Pour cette démonstration, nous utilisons un petit corpus de phrases sur deux thèmes : l'histoire de l'imprimerie et
le jardinage. Nous utilisons le package [gensim](https://radimrehurek.com/gensim/), qu'il faut installer.

> ⚠ Si Colab vous demande de redémarrer la session après l'installation de gensim, faites-le (menu **Exécution >
> Redémarrer la session**), puis réexécutez les cellules depuis le début du notebook.

In [ ]:
!pip install -q gensim

In [ ]:
from gensim import corpora
from gensim.models import LdaModel

documents = [
  "Gutenberg printed the first books with movable type in Mainz.",
  "The printing press allowed printers to produce many copies of books quickly.",
  "Early printers used paper, ink and metal type to print pages.",
  "Printed books and pamphlets spread new ideas across Europe.",
  "Gardeners plant seeds in spring and water the young plants.",
  "Tomatoes and beans grow well in a sunny garden with rich soil.",
  "Remove weeds from the garden so the plants can grow.",
  "In autumn, gardeners harvest vegetables and prepare the soil for winter.",
]

# 1. On prépare chaque document avec spaCy : on garde les lemmes des mots pleins.
textes_prepares = []
for document in documents:
  lemmes_document = []
  for token in nlp(document):
    if token.is_stop == False and token.is_alpha:
      lemmes_document.append(token.lemma_.lower())
  textes_prepares.append(lemmes_document)

print(textes_prepares)

gensim ne travaille pas directement avec des mots, mais avec des nombres :
- un `Dictionary` attribue un numéro à chaque mot du corpus ;
- `doc2bow()` transforme chaque document en « sac de mots » (*bag of words*) : la liste des numéros de ses mots, avec
  leur nombre d'occurrences. L'ordre des mots est perdu : seule compte leur fréquence.

In [ ]:
dictionnaire = corpora.Dictionary(textes_prepares)

corpus = []
for texte_prepare in textes_prepares:
  corpus.append(dictionnaire.doc2bow(texte_prepare))

print(dictionnaire.token2id)
print(corpus[0])

On peut maintenant entraîner le modèle LDA, en lui demandant de trouver 2 thèmes. `passes` est le nombre de fois où
l'algorithme parcourt le corpus, et `random_state` fixe le hasard pour obtenir toujours le même résultat.

In [ ]:
lda = LdaModel(corpus=corpus, num_topics=2, id2word=dictionnaire, passes=50, random_state=0)

for numero_theme in range(lda.num_topics):
  mots_theme = []
  for mot, poids in lda.show_topic(numero_theme, topn=6):
    mots_theme.append(mot)
  print("Thème", numero_theme, ":", mots_theme)

# Et pour chaque document, la part de chaque thème :
for document, sac_de_mots in zip(documents, corpus):
  print(lda.get_document_topics(sac_de_mots), document)

*(La fonction `zip()` permet de parcourir deux listes en même temps, élément par élément.)*

Avec un corpus aussi petit, le résultat peut être imparfait : essayez de changer `random_state` ou `passes`. Sur un vrai
corpus (des centaines d'articles de presse, de romans, de discours...), LDA peut faire émerger des thèmes très
parlants.

## 9. Tout rassembler dans une classe

Pour réutiliser facilement tout ce pipeline, rangeons-le dans une classe (voir la séance 5). Le modèle est chargé une
seule fois, dans `__init__`, car c'est l'opération la plus lente. Chaque méthode renvoie un résultat et le garde aussi
dans un attribut. En changeant le nom du modèle, la même classe fonctionne en français !

In [ ]:
class NLPMachine:
  """Applique un pipeline de TAL spaCy à un texte."""

  def __init__(self, nom_modele: str = "en_core_web_sm"):
    # "= en_core_web_sm" est une valeur par défaut : si on ne donne pas d'argument, c'est celle-ci qui est utilisée.
    self.model = spacy.load(nom_modele)
    self.text = None
    self.doc = None

  def set_text(self, texte: str):
    """Analyse un texte donné sous forme de string."""
    self.text = texte
    self.doc = self.model(texte)

  def read_text(self, path: str):
    """Lit un fichier texte et l'analyse."""
    with open(path, encoding="utf-8") as f:
      self.set_text(f.read())

  def tokenize(self):
    """Renvoie la liste des mots et la liste des phrases."""
    self.words = []
    for token in self.doc:
      self.words.append(token.text)
    self.sentences = []
    for sentence in self.doc.sents:
      self.sentences.append(sentence.text)
    return self.words, self.sentences

  def get_lemmas(self):
    """Renvoie les lemmes des mots pleins (sans mots vides ni ponctuation)."""
    self.lemmas = []
    for token in self.doc:
      if token.is_stop == False and token.is_alpha:
        self.lemmas.append(token.lemma_)
    return self.lemmas

  def get_pos(self):
    """Renvoie la catégorie grammaticale des mots pleins."""
    self.pos = []
    for token in self.doc:
      if token.is_stop == False and token.is_alpha:
        self.pos.append({"mot": token.text, "pos": token.pos_, "tag": token.tag_})
    return self.pos

  def get_ne(self):
    """Renvoie les entités nommées."""
    self.nes = []
    for entite in self.doc.ents:
      self.nes.append({"mot": entite.text, "label": entite.label_})
    return self.nes

Testons-la sur un fichier : écrivons d'abord notre texte dans `article.txt`, puis analysons-le. Puis essayons en
français, sur un texte donné directement.

In [ ]:
import os

chemin = os.path.join(os.getcwd(), "article.txt")
with open(chemin, "w", encoding="utf-8") as f:
  f.write(texte)

machine = NLPMachine()
machine.read_text(chemin)
print(machine.get_lemmas())
print(machine.get_ne())

machine_fr = NLPMachine("fr_core_news_sm")
machine_fr.set_text(texte_fr)
print(machine_fr.get_lemmas())
print(machine_fr.get_ne())

## Exercice

Ajoutez à la classe `NLPMachine` :
1. une méthode `get_frequences(n)` qui renvoie les `n` lemmes les plus fréquents ;
2. une méthode `get_sentiment()` qui renvoie la polarité et la subjectivité du texte avec TextBlob (pour l'anglais).

Puis testez votre classe sur un vrai texte : un article de Wikipédia récupéré avec `requests` et BeautifulSoup (séance 4),
ou un texte du domaine public téléchargé depuis le [Projet Gutenberg](https://www.gutenberg.org/).

Vous trouverez d'autres exercices (avec vérification automatique et corrigé) dans le notebook [Atelier Python Session 6 - Exercices.ipynb](https://colab.research.google.com/github/stage-to-data/hackspace-shs/blob/main/cours-python-resources/session-6/Atelier%20Python%20Session%206%20-%20Exercices.ipynb).

In [ ]:
# Votre code ici